# 03 — Modelagem Preditiva de Hotspots (H3 + Machine Learning)

**Projeto:** Predição Espaço-Temporal de Hotspots de Acidentes de Trânsito em Belo Horizonte
**Disciplina:** Planejamento e Gestão de Projetos (PGP) — Ciência da Computação, UFFS

Este notebook substitui o pipeline do notebook 02 e corrige os pontos levantados na revisão:

| Ponto | Notebook 02 | Este notebook |
|---|---|---|
| Recorte geográfico | Retângulo de coordenadas (inclui Contagem, Ibirité…) | Polígono oficial do município (malha IBGE) |
| Hora do acidente | `split(':')` — falha no formato `HHMMSS` | Leitura numérica `HHMMSS` |
| Período | Inclui registros esparsos de 2018–2021 | Apenas meses com cobertura regular |
| Alvo | P75 de acidentes do **próprio mês** | P90 de acidentes **por mês** (percentil recalculado a cada mês) |
| Atributos | Contagens do **mesmo mês** do alvo (vazamento) | Somente informações de **meses anteriores** |
| Divisão treino/teste | Aleatória | **Temporal** (treino no passado, teste no futuro) |
| Referência de comparação | — | Heurísticas de persistência e média móvel |
| Interpretabilidade | — | Importância por permutação + LIME |

**Pergunta respondida pelo modelo:** *com o histórico disponível até o mês anterior, este hexágono estará entre os 10% com mais acidentes de Belo Horizonte no próximo mês?*

**Entrada:** `renaest_bh_limpo.csv` (gerado no notebook 01).
**Saídas:** modelo treinado, metadados, figuras para o artigo e os arquivos que alimentam o app Streamlit.

In [ ]:
# CÉLULA 1 — Instalação e importações
!pip install -q "h3>=4" lightgbm xgboost lime

import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import joblib
import h3
from matplotlib.path import Path

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)

print('Versão do h3:', h3.__version__)
assert int(h3.__version__.split('.')[0]) >= 4, (
    'Este notebook usa a API do h3 >= 4. Use "Ambiente de execução > Reiniciar sessão" e rode de novo.'
)

In [ ]:
# CÉLULA 2 — Configurações do experimento
PASTA_DRIVE = '/content/drive/MyDrive'
ARQ_BASE = f'{PASTA_DRIVE}/renaest_bh_limpo.csv'      # saída do notebook 01
PASTA_SAIDA = f'{PASTA_DRIVE}/pgp_saidas'              # tudo o que este notebook gera

CODIGO_IBGE_BH = 3106200       # Belo Horizonte
RESOLUCAO_H3 = 8               # ~0,74 km² por hexágono
PERCENTIL_HOTSPOT = 0.90       # hotspot = 10% dos hexágonos com mais acidentes no mês
COBERTURA_MINIMA = 0.5         # mês entra no estudo se tiver >= 50% da mediana mensal de registros
MESES_AQUECIMENTO = 6          # meses iniciais usados só para formar o histórico dos atributos
INICIO_TESTE = '2025-01'       # meses a partir daqui formam o conjunto de teste
RAZAO_UNDERSAMPLING = 1.0      # negativos por positivo no treino (1.0 = classes balanceadas)
LIMIAR_DECISAO = 0.5
SEED = 42

HORAS_PICO = [7, 8, 9, 17, 18, 19]
HORAS_NOTURNAS = [22, 23, 0, 1, 2, 3, 4, 5]

In [ ]:
# CÉLULA 3 — Carregar a base limpa do notebook 01
if not os.path.exists(ARQ_BASE):
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(PASTA_SAIDA, exist_ok=True)

colunas = [
    'num_acidente', 'data_acidente', 'hora_acidente', 'fase_dia',
    'latitude_acidente', 'longitude_acidente', 'bairro_acidente',
    'qtde_envolvidos', 'qtde_obitos',
]
df = pd.read_csv(ARQ_BASE, sep=';', encoding='utf-8', usecols=colunas, low_memory=False)
print(f'Registros carregados: {len(df):,}')

In [ ]:
# CÉLULA 4 — Limite oficial de Belo Horizonte (malha municipal do IBGE)
# A API do IBGE costuma recusar conexões vindas de fora do Brasil (caso do Colab),
# então o limite é buscado primeiro no repositório do projeto, com fontes alternativas.
ARQ_LIMITE = f'{PASTA_SAIDA}/limite_bh.geojson'
FONTES_LIMITE = [
    ('repositório do projeto', 'https://raw.githubusercontent.com/matheus-marmett/'
     'PGP-Predicao-de-Risco-de-Acidentes-de-Transito-em-areas-Urbanas/main/dados/limite_bh.geojson'),
    ('geodata-br (malha IBGE de MG)', 'https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-31-mun.json'),
    ('API do IBGE', f'https://servicodados.ibge.gov.br/api/v3/malhas/municipios/{CODIGO_IBGE_BH}'
     '?formato=application/vnd.geo+json&qualidade=maxima'),
]

def extrair_bh(geojson):
    if geojson.get('type') != 'FeatureCollection':
        return geojson
    feicoes = geojson['features']
    if len(feicoes) == 1:
        return feicoes[0]['geometry']
    for feicao in feicoes:
        props = feicao.get('properties') or {}
        if str(props.get('id', props.get('codarea', ''))) == str(CODIGO_IBGE_BH):
            return feicao['geometry']
    raise ValueError('Belo Horizonte não encontrado no arquivo de limites.')

if os.path.exists(ARQ_LIMITE):
    with open(ARQ_LIMITE, encoding='utf-8') as f:
        geometria_bh = extrair_bh(json.load(f))
    print('Limite lido do cache:', ARQ_LIMITE)
else:
    geometria_bh = None
    for nome, url in FONTES_LIMITE:
        try:
            resposta = requests.get(url, timeout=20)
            resposta.raise_for_status()
            geometria_bh = extrair_bh(resposta.json())
            print('Limite obtido de:', nome)
            break
        except Exception as erro:
            print(f'Falha em {nome}: {type(erro).__name__}')
    if geometria_bh is None:
        raise RuntimeError(
            'Não foi possível baixar o limite de BH. Envie o arquivo dados/limite_bh.geojson do repositório '
            f'para {ARQ_LIMITE} (painel Arquivos do Colab) e rode esta célula de novo.'
        )
    with open(ARQ_LIMITE, 'w', encoding='utf-8') as f:
        json.dump({'type': 'FeatureCollection',
                   'features': [{'type': 'Feature', 'properties': {'id': str(CODIGO_IBGE_BH)},
                                 'geometry': geometria_bh}]}, f)

def aneis_externos(geom):
    if geom['type'] == 'Polygon':
        return [geom['coordinates'][0]]
    if geom['type'] == 'MultiPolygon':
        return [poligono[0] for poligono in geom['coordinates']]
    raise ValueError(f"Geometria não suportada: {geom['type']}")

CONTORNOS_BH = [Path(np.asarray(anel, dtype=float)) for anel in aneis_externos(geometria_bh)]

def dentro_de_bh(lat, lng):
    # Coordenadas GeoJSON são (longitude, latitude)
    pontos = np.column_stack([np.asarray(lng, dtype=float), np.asarray(lat, dtype=float)])
    resultado = np.zeros(len(pontos), dtype=bool)
    for contorno in CONTORNOS_BH:
        resultado |= contorno.contains_points(pontos)
    return resultado

print('Limite de BH carregado:', geometria_bh['type'], f'({len(CONTORNOS_BH)} contorno(s))')

In [ ]:
# CÉLULA 5 — Limpeza: duplicatas, recorte municipal, datas, hora e período
relatorio = {'registros_iniciais': len(df)}

# 1. Duplicatas
df = df.drop_duplicates(subset='num_acidente')
relatorio['apos_remover_duplicatas'] = len(df)

# 2. Coordenadas e recorte pelo polígono do município
for col in ['latitude_acidente', 'longitude_acidente']:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce')
df = df.dropna(subset=['latitude_acidente', 'longitude_acidente'])
df = df[dentro_de_bh(df['latitude_acidente'], df['longitude_acidente'])].copy()
relatorio['dentro_do_municipio'] = len(df)

# 3. Datas
df['data'] = pd.to_datetime(df['data_acidente'], errors='coerce')
df = df.dropna(subset=['data'])
df['periodo'] = df['data'].dt.to_period('M')

# 4. Hora no formato numérico HHMMSS (ex.: 170000 = 17:00:00; 90000 = 09:00:00)
hora_num = pd.to_numeric(df['hora_acidente'], errors='coerce')
hora = hora_num // 10000
minuto = (hora_num // 100) % 100
hora_valida = hora_num.notna() & hora.between(0, 23) & minuto.between(0, 59)
df['hora'] = hora.where(hora_valida)
relatorio['hora_valida_pct'] = round(100 * hora_valida.mean(), 2)

# Sem hora válida, usa a fase do dia informada no registro
fase = df['fase_dia'].astype(str).str.upper().str.strip()
df['pico'] = df['hora'].isin(HORAS_PICO).astype(int)
df['noturno'] = np.where(df['hora'].notna(),
                         df['hora'].isin(HORAS_NOTURNAS),
                         fase.isin(['NOITE', 'MADRUGADA'])).astype(int)
df['fds'] = (df['data'].dt.dayofweek >= 5).astype(int)

for col in ['qtde_envolvidos', 'qtde_obitos']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
df['com_obito'] = (df['qtde_obitos'] > 0).astype(int)

# 5. Período com cobertura regular de registros
contagem_mensal = df.groupby('periodo').size()
minimo = COBERTURA_MINIMA * contagem_mensal.median()
meses_ok = contagem_mensal[contagem_mensal >= minimo].index
INICIO, FIM = meses_ok.min(), meses_ok.max()
lacunas = [str(p) for p in pd.period_range(INICIO, FIM, freq='M') if contagem_mensal.get(p, 0) < minimo]
df = df[(df['periodo'] >= INICIO) & (df['periodo'] <= FIM)].copy()
relatorio['no_periodo_estudado'] = len(df)
relatorio['periodo'] = f'{INICIO} a {FIM}'

for chave, valor in relatorio.items():
    print(f'{chave:>28}: {valor:,}' if isinstance(valor, int) else f'{chave:>28}: {valor}')
if lacunas:
    print('⚠️ Meses dentro do período com cobertura abaixo do mínimo:', lacunas)
print(f"\nAcidentes com óbito: {df['com_obito'].sum():,} ({100 * df['com_obito'].mean():.2f}%)")

In [ ]:
# CÉLULA 6 — Indexação espacial H3 (resolução 8)
df['h3'] = [h3.latlng_to_cell(lat, lng, RESOLUCAO_H3)
            for lat, lng in zip(df['latitude_acidente'], df['longitude_acidente'])]

# Todos os hexágonos de BH (inclusive os que nunca registraram acidente)
hex_municipio = set(h3.geo_to_cells(geometria_bh, RESOLUCAO_H3))
HEXAGONOS = sorted(hex_municipio | set(df['h3']))

print(f'Hexágonos com centro dentro de BH: {len(hex_municipio):,}')
print(f'Hexágonos no estudo (incluindo bordas com acidentes): {len(HEXAGONOS):,}')
print(f"Hexágonos que registraram ao menos um acidente: {df['h3'].nunique():,}")

In [ ]:
# CÉLULA 7 — Painel hexágono × mês (inclui meses sem acidentes e o mês a prever)
MES_PREVISTO = FIM + 1
meses = pd.period_range(INICIO, MES_PREVISTO, freq='M')

painel = pd.MultiIndex.from_product([HEXAGONOS, meses], names=['h3', 'periodo']).to_frame(index=False)

agregado = df.groupby(['h3', 'periodo']).agg(
    acidentes=('num_acidente', 'size'),
    obitos=('qtde_obitos', 'sum'),
    acid_com_obito=('com_obito', 'sum'),
    envolvidos=('qtde_envolvidos', 'sum'),
    pico=('pico', 'sum'),
    noturnas=('noturno', 'sum'),
    fds=('fds', 'sum'),
).reset_index()

painel = painel.merge(agregado, on=['h3', 'periodo'], how='left')
COLS_CONTAGEM = ['acidentes', 'obitos', 'acid_com_obito', 'envolvidos', 'pico', 'noturnas', 'fds']
painel[COLS_CONTAGEM] = painel[COLS_CONTAGEM].fillna(0)
painel.loc[painel['periodo'] == MES_PREVISTO, COLS_CONTAGEM] = np.nan   # futuro: ainda desconhecido
painel = painel.sort_values(['h3', 'periodo']).reset_index(drop=True)

print(f'Painel: {len(HEXAGONOS):,} hexágonos × {len(meses)} meses = {len(painel):,} linhas')
print(f'Mês a prever (fora dos dados): {MES_PREVISTO}')

In [ ]:
# CÉLULA 8 — Variável-alvo: hotspot = P90 de acidentes entre os hexágonos de BH, mês a mês
observado = painel['periodo'] <= FIM
limiar_mes = painel[observado].groupby('periodo')['acidentes'].quantile(PERCENTIL_HOTSPOT)
painel['limiar_p90'] = painel['periodo'].map(limiar_mes)

painel['is_hotspot'] = ((painel['acidentes'] >= painel['limiar_p90']) & (painel['acidentes'] > 0)).astype(float)
painel.loc[~observado, 'is_hotspot'] = np.nan

prop = painel[observado].groupby('periodo')['is_hotspot'].mean()
print(f'Limiar P90 por mês: mín {limiar_mes.min():.0f} | mediana {limiar_mes.median():.0f} | máx {limiar_mes.max():.0f} acidentes')
print(f'Proporção de hotspots por mês: média {prop.mean():.1%} (mín {prop.min():.1%}, máx {prop.max():.1%})')

fig, ax = plt.subplots(figsize=(11, 3.5))
limiar_mes.plot(ax=ax, marker='o')
ax.set_title('Limiar mensal de hotspot (P90 de acidentes por hexágono)')
ax.set_xlabel('Mês'); ax.set_ylabel('Acidentes no hexágono')
plt.tight_layout(); plt.savefig(f'{PASTA_SAIDA}/fig_limiar_p90.png', dpi=150); plt.show()

In [ ]:
# CÉLULA 9 — Atributos preditivos (somente meses ANTERIORES ao mês do alvo)
g = painel.groupby('h3', sort=False)

def passado(coluna, janela, agregacao='mean'):
    # Janela móvel terminando no mês anterior (shift(1) impede o uso do próprio mês)
    return g[coluna].transform(lambda s: s.shift(1).rolling(janela, min_periods=1).agg(agregacao))

def razao(numerador, denominador):
    return (numerador / denominador.replace(0, np.nan)).fillna(0)

painel['acid_lag1'] = g['acidentes'].shift(1)
painel['acid_media_3m'] = passado('acidentes', 3)
painel['acid_media_6m'] = passado('acidentes', 6)
painel['acid_media_12m'] = passado('acidentes', 12)
painel['acid_tendencia'] = painel['acid_media_3m'] - painel['acid_media_12m']
painel['hotspot_lag1'] = g['is_hotspot'].shift(1)
painel['hotspot_freq_12m'] = passado('is_hotspot', 12)

acid_12m = passado('acidentes', 12, 'sum')
painel['obitos_12m'] = passado('obitos', 12, 'sum')
painel['envolvidos_por_acid_12m'] = razao(passado('envolvidos', 12, 'sum'), acid_12m)
painel['prop_pico_12m'] = razao(passado('pico', 12, 'sum'), acid_12m)
painel['prop_noturnas_12m'] = razao(passado('noturnas', 12, 'sum'), acid_12m)
painel['prop_fds_12m'] = razao(passado('fds', 12, 'sum'), acid_12m)

# Vizinhança: média móvel de 3 meses dos hexágonos adjacentes (anel k = 1)
universo = set(HEXAGONOS)
arestas = pd.DataFrame(
    [(c, v) for c in HEXAGONOS for v in h3.grid_disk(c, 1) if v != c and v in universo],
    columns=['h3', 'vizinho'],
)
media_vizinhos = (
    arestas.merge(painel[['h3', 'periodo', 'acid_media_3m']]
                  .rename(columns={'h3': 'vizinho', 'acid_media_3m': 'valor'}), on='vizinho')
    .groupby(['h3', 'periodo'])['valor'].mean()
    .rename('vizinhos_media_3m').reset_index()
)
painel = painel.merge(media_vizinhos, on=['h3', 'periodo'], how='left')

# Sazonalidade cíclica do mês
mes_num = painel['periodo'].dt.month
painel['mes_sin'] = np.sin(2 * np.pi * mes_num / 12)
painel['mes_cos'] = np.cos(2 * np.pi * mes_num / 12)

painel['meses_historico'] = painel.groupby('h3').cumcount()

FEATURES = [
    'acid_lag1', 'acid_media_3m', 'acid_media_6m', 'acid_media_12m', 'acid_tendencia',
    'hotspot_lag1', 'hotspot_freq_12m', 'obitos_12m', 'envolvidos_por_acid_12m',
    'prop_pico_12m', 'prop_noturnas_12m', 'prop_fds_12m', 'vizinhos_media_3m',
    'mes_sin', 'mes_cos',
]
painel[FEATURES] = painel[FEATURES].fillna(0)

base_modelo = painel[(painel['periodo'] <= FIM) & (painel['meses_historico'] >= MESES_AQUECIMENTO)].copy()
print(f'{len(FEATURES)} atributos | amostras para modelagem: {len(base_modelo):,}')
print(f"Primeiro mês com alvo modelado: {base_modelo['periodo'].min()} (após {MESES_AQUECIMENTO} meses de aquecimento)")
base_modelo[FEATURES].describe().T.round(3)

In [ ]:
# CÉLULA 10 — Divisão temporal e subamostragem (somente no treino)
corte = pd.Period(INICIO_TESTE, freq='M')
treino = base_modelo[base_modelo['periodo'] < corte]
teste = base_modelo[base_modelo['periodo'] >= corte]
assert len(treino) and len(teste), f'Ajuste INICIO_TESTE: os dados vão de {INICIO} a {FIM}.'

X_teste, y_teste = teste[FEATURES], teste['is_hotspot'].astype(int)

positivos = treino[treino['is_hotspot'] == 1]
negativos = treino[treino['is_hotspot'] == 0]
n_neg = min(len(negativos), int(round(len(positivos) * RAZAO_UNDERSAMPLING)))
treino_bal = pd.concat([positivos, negativos.sample(n=n_neg, random_state=SEED)]).sample(frac=1, random_state=SEED)
X_treino, y_treino = treino_bal[FEATURES], treino_bal['is_hotspot'].astype(int)

def resumo(nome, dados):
    print(f"{nome:<22} {dados['periodo'].min()} a {dados['periodo'].max()} | "
          f"{len(dados):>7,} amostras | hotspots: {dados['is_hotspot'].mean():.1%}")

resumo('Treino (original)', treino)
resumo('Treino (subamostrado)', treino_bal)
resumo('Teste', teste)

In [ ]:
# CÉLULA 11 — Treinamento e avaliação no conjunto de teste (meses futuros)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

modelos = {
    'Regressão Logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=SEED)),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=20,
                                            n_jobs=-1, random_state=SEED),
    'XGBoost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8,
                             colsample_bytree=0.8, eval_metric='logloss', random_state=SEED),
    'LightGBM': LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, subsample=0.8,
                               subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1),
}

def metricas(nome, y_true, pred, score):
    return {
        'Modelo': nome,
        'Acurácia': accuracy_score(y_true, pred),
        'Precisão': precision_score(y_true, pred, zero_division=0),
        'Recall': recall_score(y_true, pred, zero_division=0),
        'F1-Score': f1_score(y_true, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, score),
        'PR-AUC': average_precision_score(y_true, score),
    }

resultados, probabilidades = [], {}

# Referências simples: qualquer modelo útil precisa superá-las
persistencia = teste['hotspot_lag1'].astype(int)
resultados.append(metricas('Heurística: hotspot no mês anterior', y_teste, persistencia, teste['hotspot_lag1']))
limiar_anterior = teste['periodo'].map(lambda p: limiar_mes.get(p - 1, np.nan))
media_movel = (teste['acid_media_3m'] >= limiar_anterior).astype(int)
resultados.append(metricas('Heurística: média 3 meses ≥ P90 anterior', y_teste, media_movel, teste['acid_media_3m']))

for nome, modelo in modelos.items():
    modelo.fit(X_treino, y_treino)
    proba = modelo.predict_proba(X_teste)[:, 1]
    probabilidades[nome] = proba
    resultados.append(metricas(nome, y_teste, (proba >= LIMIAR_DECISAO).astype(int), proba))

tabela = pd.DataFrame(resultados).set_index('Modelo').round(4)
tabela_ml = tabela.loc[list(modelos)].sort_values('PR-AUC', ascending=False)
MELHOR = tabela_ml.index[0]
melhor_modelo = modelos[MELHOR]

print('=== DESEMPENHO NO TESTE (meses a partir de', INICIO_TESTE, ') ===')
print(tabela.sort_values('PR-AUC', ascending=False).to_string())
print(f'\nMelhor modelo (critério PR-AUC, adequado a classes desbalanceadas): {MELHOR}')

In [ ]:
# CÉLULA 12 — Matriz de confusão e importância dos atributos do melhor modelo
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

pred_melhor = (probabilidades[MELHOR] >= LIMIAR_DECISAO).astype(int)
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_teste, pred_melhor),
                       display_labels=['Não hotspot', 'Hotspot']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de confusão — {MELHOR}')
plt.tight_layout(); plt.savefig(f'{PASTA_SAIDA}/fig_matriz_confusao.png', dpi=150); plt.show()

imp = permutation_importance(melhor_modelo, X_teste, y_teste, scoring='average_precision',
                             n_repeats=10, random_state=SEED, n_jobs=-1)
importancia = pd.Series(imp.importances_mean, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7, 5))
importancia.plot.barh(ax=ax)
ax.set_title(f'Importância por permutação (queda de PR-AUC) — {MELHOR}')
plt.tight_layout(); plt.savefig(f'{PASTA_SAIDA}/fig_importancia_atributos.png', dpi=150); plt.show()
print(importancia.sort_values(ascending=False).round(4).to_string())

In [ ]:
# CÉLULA 13 — Explicabilidade local com LIME (hotspot corretamente previsto no teste)
from lime.lime_tabular import LimeTabularExplainer

explicador = LimeTabularExplainer(
    X_treino.values, feature_names=FEATURES, class_names=['Não hotspot', 'Hotspot'],
    mode='classification', discretize_continuous=True, random_state=SEED,
)
prever = lambda x: melhor_modelo.predict_proba(pd.DataFrame(x, columns=FEATURES))

acertos = teste.assign(proba=probabilidades[MELHOR])
acertos = acertos[acertos['is_hotspot'] == 1]
if (acertos['proba'] >= LIMIAR_DECISAO).any():
    acertos = acertos[acertos['proba'] >= LIMIAR_DECISAO]
exemplo = acertos.sort_values('proba', ascending=False).iloc[len(acertos) // 2]   # caso típico, não o extremo

explicacao = explicador.explain_instance(exemplo[FEATURES].astype(float).values, prever, num_features=8)
print(f"Hexágono {exemplo['h3']} em {exemplo['periodo']} — probabilidade prevista: {exemplo['proba']:.1%}")
for regra, peso in explicacao.as_list():
    print(f'  {peso:+.3f}  {regra}')
fig = explicacao.as_pyplot_figure()
plt.title('LIME — fatores da previsão de hotspot'); plt.tight_layout()
plt.savefig(f'{PASTA_SAIDA}/fig_lime_exemplo.png', dpi=150); plt.show()

In [ ]:
# CÉLULA 14 — Exportar modelo, metadados e dados para o app Streamlit
ARQ_MODELO = f'{PASTA_SAIDA}/modelo_hotspot_bh_h3.joblib'
ARQ_META = f'{PASTA_SAIDA}/metadados_modelo.json'
ARQ_HEX = f'{PASTA_SAIDA}/hexagonos_bh.csv'
ARQ_HIST = f'{PASTA_SAIDA}/historico_mensal_bh.csv'

joblib.dump(melhor_modelo, ARQ_MODELO, compress=3)

# Previsão para o mês seguinte ao último mês com dados
futuro = painel[painel['periodo'] == MES_PREVISTO].copy()
futuro['probabilidade'] = melhor_modelo.predict_proba(futuro[FEATURES])[:, 1]
futuro['previsao_hotspot'] = (futuro['probabilidade'] >= LIMIAR_DECISAO).astype(int)
futuro['ranking_risco'] = futuro['probabilidade'].rank(ascending=False, method='min').astype(int)

centros = {c: h3.cell_to_latlng(c) for c in HEXAGONOS}
futuro['lat'] = futuro['h3'].map(lambda c: centros[c][0])
futuro['lng'] = futuro['h3'].map(lambda c: centros[c][1])

# Bairro mais frequente nos registros de cada hexágono (para busca no app)
nome_bairro = df['bairro_acidente'].astype(str).str.strip().str.upper()
validos = ~nome_bairro.isin(['', 'NAN', 'NONE', 'NAO INFORMADO', 'NÃO INFORMADO'])
bairro = (pd.DataFrame({'h3': df['h3'], 'bairro': nome_bairro})[validos]
            .value_counts().reset_index(name='n')
            .sort_values(['h3', 'n'], ascending=[True, False])
            .drop_duplicates('h3').set_index('h3')['bairro'])
futuro['bairro'] = futuro['h3'].map(bairro).fillna('Sem registro')

totais = df.groupby('h3').agg(acidentes_total=('num_acidente', 'size'), obitos_total=('qtde_obitos', 'sum'))
futuro = futuro.merge(totais, left_on='h3', right_index=True, how='left').fillna({'acidentes_total': 0, 'obitos_total': 0})
futuro['mes_previsto'] = str(MES_PREVISTO)

colunas_app = ['h3', 'bairro', 'lat', 'lng', 'mes_previsto', 'probabilidade', 'previsao_hotspot',
               'ranking_risco', 'acidentes_total', 'obitos_total'] + FEATURES
futuro[colunas_app].sort_values('ranking_risco').to_csv(ARQ_HEX, index=False)

historico = painel.loc[painel['periodo'] <= FIM, ['h3', 'periodo', 'acidentes', 'obitos', 'is_hotspot']].copy()
historico['periodo'] = historico['periodo'].astype(str)
historico['is_hotspot'] = historico['is_hotspot'].astype(int)
historico.to_csv(ARQ_HIST, index=False)

metadados = {
    'modelo': MELHOR,
    'features': FEATURES,
    'resolucao_h3': RESOLUCAO_H3,
    'percentil_hotspot': PERCENTIL_HOTSPOT,
    'limiar_decisao': LIMIAR_DECISAO,
    'periodo_dados': [str(INICIO), str(FIM)],
    'inicio_teste': INICIO_TESTE,
    'mes_previsto': str(MES_PREVISTO),
    'n_hexagonos': len(HEXAGONOS),
    'n_acidentes': int(len(df)),
    'metricas_teste': tabela.reset_index().to_dict(orient='records'),
}
with open(ARQ_META, 'w', encoding='utf-8') as f:
    json.dump(metadados, f, ensure_ascii=False, indent=2)

print(f"Previsão para {MES_PREVISTO}: {futuro['previsao_hotspot'].sum()} hexágonos classificados como hotspot")
print(futuro[['h3', 'bairro', 'probabilidade', 'acid_media_3m']].sort_values('probabilidade', ascending=False).head(10).to_string(index=False))
for arq in [ARQ_MODELO, ARQ_META, ARQ_HEX, ARQ_HIST]:
    print(f'Salvo: {arq} ({os.path.getsize(arq) / 1024:.1f} KB)')

In [ ]:
# CÉLULA 15 — Baixar os arquivos para enviar ao repositório
try:
    from google.colab import files
    for arq in [ARQ_MODELO, ARQ_META, ARQ_HEX, ARQ_HIST]:
        files.download(arq)
except ImportError:
    print('Fora do Colab: os arquivos estão em', PASTA_SAIDA)

In [ ]:
# CÉLULA 16 — Resumo para o artigo
n_hot_treino = int(treino['is_hotspot'].sum())
print(f'''
PERÍODO ANALISADO: {INICIO} a {FIM} ({len(pd.period_range(INICIO, FIM, freq="M"))} meses)
ACIDENTES EM BH (após limpeza): {len(df):,}
HEXÁGONOS H3 (res. {RESOLUCAO_H3}): {len(HEXAGONOS):,}
AMOSTRAS (hexágono × mês) modeladas: {len(base_modelo):,}
TREINO: {treino["periodo"].min()} a {treino["periodo"].max()} — {len(treino):,} amostras ({n_hot_treino:,} hotspots); após subamostragem: {len(treino_bal):,}
TESTE:  {teste["periodo"].min()} a {teste["periodo"].max()} — {len(teste):,} amostras ({teste["is_hotspot"].mean():.1%} hotspots)
MELHOR MODELO (PR-AUC): {MELHOR}
''')
print(tabela.sort_values('PR-AUC', ascending=False).to_markdown())